In [69]:
import xarray as xr
import os
import geopandas as gpd
import xarray as xr
import geopandas as gpd
import pandas as pd
import juliandate as jd
from multiprocessing import Pool

In [70]:
 # load all files
os.chdir("/Users/anora/Team MG Dropbox/Wanru Wu/Cloudseeding_Anora/SSF/raw/terra")
path = os.getcwd() 

# Get the list of all files and directories 
all_files = os.listdir(path) 
data_files_terra = [file for file in all_files if file.find("CERES_SSF_Terra-XTRK_Edition4A")!=-1]

In [71]:
# Load the county-level district file
county_gdf = gpd.read_file(os.path.dirname(os.getcwd()) + "/district/district.shp")
county_gdf = county_gdf.to_crs("EPSG:4326")

In [72]:
with xr.open_dataset(data_files_terra[1]) as ds:
    ds


In [ ]:
ds

<xarray.Dataset>
Dimensions:                                                        (
                                                                    time: 3763258,
                                                                    The_8_most_prevalent_surface_types: 8,
                                                                    Conditions_clear__lower__upper__upper_over_lower: 4,
                                                                    Lower_and_Upper_Cloud_Layers: 2)
Coordinates:
  * time                                                           (time) datetime64[ns] ...
Dimensions without coordinates: The_8_most_prevalent_surface_types,
                                Conditions_clear__lower__upper__upper_over_lower,
                                Lower_and_Upper_Cloud_Layers
Data variables: (12/86)
    lon                                                            (time) float32 ...
    lat                                                            (time) float32 ...
    Time_of_observation                                            (time) float64 ...
    Longitude_of_CERES_FOV_at_surface                              (time) float32 ...
    Colatitude_of_CERES_FOV_at_surface                             (time) float32 ...
    Longitude_of_subsatellite_point_at_surface_at_observation      (time) float32 ...
    ...                                                             ...
    Percentage_of_CERES_FOV_with_MODIS_deep_blue_aerosol           (time) float32 ...
    PSF_wtd_MOD04_deep_blue_aerosol_optical_depth_land__0_550_     (time) float32 ...
    PSF_wtd_MOD04_deep_blue_angstrom_exponent_land                 (time) float32 ...
    PSF_wtd_MOD04_deep_blue_single_scattering_albedo_land__0_470_  (time) float32 ...
    Packet_number                                                  (time) float32 ...
    Scan_sample_number                                             (time) float32 ...
Attributes:
    Conventions:                CF-1.0
    Subsetter_title:            ASDC CERES Subset
    Subsetter_version:          2.9.b1
    Subsetter_institution:      Atmospheric Science Data Center (ASDC) http:/...
    Subsetter_history:          2025-05-31T21:12:07 -0400 SubsetCeresSsf
    Subsetter_temporalFilter:   2011-01-01T00:00:00.000000Z to 2011-12-31T23:...
    Subsetter_spatialFilter:    POLYGON ((73 18, 73 54, 135 54, 135 18, 73 18))
    Subsetter_parameterFilter:  none
    platform:                   Terra
    history_json:               [{"$schema": "https://harmony.earthdata.nasa....

In [74]:
districts = county_gdf

lons = ds["lon"].values
lats = ds["lat"].values

# Create GeoDataFrame of points
point_gdf = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(lons, lats),
    crs="EPSG:4326"
)

# Spatial join - extract county code
joined = gpd.sjoin(point_gdf, districts, how="left", predicate="within")

# Extract date
dates = pd.to_datetime(ds["time"].values).tz_localize("UTC").tz_convert("Asia/Shanghai").tz_localize(None).values.astype("datetime64[D]").astype("U10")
ds["date"] = xr.DataArray(dates, dims="time")

# Add back to the netcdf
ds["region"] = xr.DataArray(joined["dt_adcode"].to_numpy(), dims="time")
print("added region and time variable")

# Filter valid rows
cat1 = pd.Categorical(ds["region"].values)
mask = (cat1.codes >= 0) 
ds_masked = ds.isel(time=mask)  
print("masked")

# Created combined group of region and time
region_vals = ds_masked["region"].values
date_vals = ds_masked["date"].values
combined_labels = [f"{r}_{d}" for r, d in zip(region_vals, date_vals)]
ds_masked["combined_group"] = xr.DataArray(combined_labels, dims=["time"])
print("assigned combined group")


added region and time variable
masked
assigned combined group


In [80]:
# Select variables to keep
variables_to_keep = ['CERES_SW_TOA_flux___upwards',
                     'CERES_solar_zenith_at_surface',
                     'date',
                     'region']
ds_selected = ds_masked[variables_to_keep]
print("selected target variables")

# Convert to dataframe
df_temp = ds_selected.to_dataframe()
print("converted")


selected target variables
converted


In [82]:
max(df_temp.CERES_solar_zenith_at_surface)

143.2674102783203

In [87]:
test3 = df_temp[df_temp['CERES_solar_zenith_at_surface']>90]
test3.head(500)

,CERES_SW_TOA_flux___upwards,CERES_solar_zenith_at_surface,date,region
time,,,,
2011-07-02 12:37:56.923075328,0.0,103.714172,2011-07-02,220623
2011-07-02 12:37:56.963067136,0.0,103.269844,2011-07-02,220582
2011-07-02 12:37:58.523066624,0.0,103.222809,2011-07-02,220582
2011-07-02 12:37:58.563098624,0.0,103.653923,2011-07-02,220681
2011-07-02 12:38:03.453074176,0.0,103.929924,2011-07-02,222405
2011-07-02 12:38:03.473070080,0.0,103.780579,2011-07-02,222406
2011-07-02 12:38:03.493065728,0.0,103.617249,2011-07-02,222406
2011-07-02 12:38:03.513061632,0.0,103.439713,2011-07-02,220621
2011-07-02 12:38:03.533057536,0.0,103.243523,2011-07-02,220681


In [77]:
test = df_temp[df_temp['region']=="232721"]
test = test[test['date']=="2012-01-01"]
pd.set_option('display.max_rows', 500)
test.head(200)

,CERES_SW_TOA_flux___upwards,CERES_viewing_zenith_at_surface,date,region
time,,,,


In [78]:
test2 = df_temp[df_temp['region']=="232721"]
test2 = test2[test2['date']=="2012-01-02"]
pd.set_option('display.max_rows', 500)
test2.head(500)

,CERES_SW_TOA_flux___upwards,CERES_viewing_zenith_at_surface,date,region
time,,,,


In [79]:
test2 = df_temp[df_temp['region']=="310114"]
test2 = test2[test2['date']=="2012-02-05"]
pd.set_option('display.max_rows', 500)
test2.head(500)

,CERES_SW_TOA_flux___upwards,CERES_viewing_zenith_at_surface,date,region
time,,,,


In [ ]:
import xarray as xr
with xr.open_dataset("/Users/anora/Desktop/MCD06COSP_D3_MODIS.A2016001.062.2022124091724.nc") as ds:
    ds
ds

<xarray.Dataset>
Dimensions:    (latitude: 180, longitude: 360)
Coordinates:
  * latitude   (latitude) float64 -89.5 -88.5 -87.5 -86.5 ... 87.5 88.5 89.5
  * longitude  (longitude) float64 -179.5 -178.5 -177.5 ... 177.5 178.5 179.5
Data variables:
    *empty*
Attributes: (12/51)
    YAML_config:                       grid_settings:\n  gridsize: 1\n  proje...
    Yori_version:                      1.5.0
    input_files:                       MCD06COSP_G3_MODIS_Aqua.A2016001.0000....
    daily_defn_of_day_adjustment:      False
    history:                           
    source:                            idl 8.4, mcd06cosp_preyori 20220218-1,...
    ...                                ...
    longitude_resolution:              1.0
    license:                           http://science.nasa.gov/earth-science/...
    stdname_vocabulary:                NetCDF Climate and Forecast (CF) Metad...
    keywords_vocabulary:               NASA Global Change Master Directory (G...
    keywords:                          EARTH SCIENCE > ATMOSPHERE > CLOUDS > ...
    naming_authority:                  gov.nasa.gsfc.sci.atmos

In [6]:
import netCDF4 as nc
ncf = nc.Dataset("/Users/anora/Desktop/MCD06COSP_D3_MODIS.A2016001.062.2022124091724.nc")

for group_name in ncf.groups:
    group = ncf.groups[group_name]

thickness_data = ncf.groups["Cloud_Optical_Thickness_Total"] 

fraction_data= ncf.groups['Cloud_Mask_Fraction']
fraction_data

<class 'netCDF4.Group'>
group /Cloud_Mask_Fraction:
    long_name: Cloud Fraction from Cloud Mask for Daytime Scenes
    units: none
    _FillValue: -999.0
    valid_min: 0.0
    valid_max: 1.0
    scale_factor: 1.0
    add_offset: 0.0
    dimensions(sizes): 
    variables(dimensions): float64 Mean(longitude, latitude), float64 Standard_Deviation(longitude, latitude), float64 Sum(longitude, latitude), int32 Pixel_Counts(longitude, latitude), float64 Sum_Squares(longitude, latitude)
    groups: 